## 05 — Temporary tables
Show applied policy/revision, bounded enabled windows, expiry, and cleanup markers.

In [ ]:
import clickhouse_connect, os
client = clickhouse_connect.get_client(
    host=os.environ.get('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.environ.get('CLICKHOUSE_PORT', '8123')),
    username=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', ''),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'market'),
)
print('clickhouse', client.server_version)


In [ ]:
status = client.query(
    "SELECT table_name, policy, row_ttl_ns, projection_revision "
    "FROM market._recording_status FINAL ORDER BY table_name"
).result_rows
print(status)
assert status, 'recording status table missing'

declared = {row[0] for row in status}
permanent = {row[0] for row in status if row[1] == 'permanent'}
temporary = {row[0] for row in status if row[1] == 'temporary'}

# The recorder declares exactly these ten tables; the status row set must
# match, so a dropped declaration shows up here rather than as a silently
# missing table.
expected = {
    'trades', 'quotes', 'raw_exchange_messages', 'sbe_messages', 'l2_books',
    'order_book_deltas', 'funding_rates', 'mark_prices', 'index_prices',
    'book_debug',
}
assert declared == expected, f'declared tables differ: {sorted(declared ^ expected)}'
assert temporary == {'book_debug'}, f'unexpected temporary tables: {sorted(temporary)}'
assert permanent == expected - temporary, f'unexpected permanent tables: {sorted(permanent)}'
# Every row must be classifiable — an unknown policy would fall out of both
# sets and go unverified.
assert len(permanent) + len(temporary) == len(declared), 'unclassified policy value'
print('permanent:', sorted(permanent))
print('temporary:', sorted(temporary))

In [ ]:
dbg = client.query('SELECT count() FROM book_debug FINAL').result_rows[0][0]
print('book_debug rows:', dbg)
# A temporary table that was never enabled holds no rows. The status table
# above is the evidence that it was declared rather than missing, so an empty
# table is a recorded disabled state and not a dashboard-wide SQL error.
# `isinstance` would be unconditionally true (a count is always an int); the
# real check is that the table exists at all and answers with a count.
assert isinstance(dbg, int), 'book_debug is not queryable'
assert dbg >= 0, 'book_debug returned a negative count'
exists = client.query("SELECT count() FROM system.tables WHERE database = currentDatabase() AND name = 'book_debug'").result_rows[0][0]
assert exists == 1, 'book_debug is not declared in the schema'
print('book_debug declared and queryable')